In [ ]:
import glob, os, shutil, zipfile, json, numpy as np, pandas as pd

ROOT = '/kaggle/input/competitions/rsna-knee-abnormality-detection'
UIDS = pd.read_csv(f'{ROOT}/train.csv').StudyInstanceUID.tolist()
CHUNK, K, S = 256, 24, 192
DST = '/kaggle/working/shards'
os.makedirs(DST, exist_ok=True)

src = {os.path.basename(f): f for f in glob.glob('/kaggle/input/**/shard_*.npz', recursive=True)}
assert len(src) == 18, f"expected 18 unique shards, found {len(src)}"

for name, f in sorted(src.items()):
    shutil.copy(f, f'{DST}/{name}')

# verify the COPIES, not the sources
seen = set()
for name in sorted(os.listdir(DST)):
    p = f'{DST}/{name}'
    ci = int(name.split('_')[1].split('.')[0])
    assert zipfile.ZipFile(p).testzip() is None, f"CRC fail {name}"
    d = np.load(p); x, m, uid = d['x'], d['m'], d['uid']
    exp = UIDS[ci*CHUNK:(ci+1)*CHUNK]
    assert x.shape == (len(exp), 3, K, S, S) and x.dtype == np.uint8, (name, x.shape)
    assert list(uid) == exp, f"uid mismatch {name}"
    assert (m.sum(1) == 0).sum() == 0, f"empty studies {name}"
    seen |= set(uid)
assert len(seen) == len(UIDS) == 4407, len(seen)

json.dump({"n_shards": 18, "n_studies": len(seen), "K": K, "S": S, "chunk": CHUNK},
          open(f'{DST}/manifest.json', 'w'))
print("ALL GOOD:", len(os.listdir(DST)) - 1, "shards,", len(seen), "studies,",
      round(sum(os.path.getsize(f'{DST}/{n}') for n in os.listdir(DST)) / 1e9, 1), "GB")